# Flow Assurance and Pipeline Studies

13 September 2026 revision. The numerical examples use a synthetic teaching basis. Conceptual illustrations represent workflow relationships; they are not measured performance data. The image-generator art is retained with prompts and hashes in `illustrations/imagegen_manifest_2026-09-13.json`; this notebook checks and restores the reviewed raster asset rather than regenerating it through an AI call. Other diagrams and numerical plots are drawn by the maintained illustration code. Set `NEQSIM_PROJECT_ROOT` to the recorded source checkout before running numerical cells.

In [1]:
import os
import sys
from pathlib import Path

start = Path.cwd().resolve()
BOOK_DIR = next(p for p in [start] + list(start.parents)
                if (p / "book.yaml").is_file() and (p / "book_runtime.py").is_file())
sys.path.insert(0, str(BOOK_DIR))
from book_runtime import bootstrap
from build_illustrations import generate_chapter
import json
results = json.loads((BOOK_DIR / "results.json").read_text(encoding="utf-8"))
baseline_record = json.loads((BOOK_DIR / "verification" / "regression_baseline_2026-09-12.json").read_text(encoding="utf-8"))
baselines = baseline_record["outputs"]
assert results["basis"] == baseline_record["basis"], "Review a changed case basis before updating accepted baselines"


In [2]:
jneqsim = bootstrap(os.environ["NEQSIM_PROJECT_ROOT"])
from verify_examples import make_fluid, compression_case, pipeline_case

NeqSim project root: C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim\target\classes
  2. C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim\target\neqsim-3.20.0.jar



JVM started: C:\Program Files\Java\graalvm-25.3.4.1+1.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


In [3]:
import numpy as np
for expected in baselines["pipeline_sensitivity"]:
    row = pipeline_case(jneqsim, expected["diameter_m"], expected["increments"])
    assert abs(row["pressure_drop_bar"] - expected["pressure_drop_bar"]) < 1e-6
    assert abs(row["outlet_temperature_C"] - expected["outlet_temperature_C"]) < 1e-6
    assert np.allclose(row["pressure_profile_bara"], expected["pressure_profile_bara"], atol=1e-6, rtol=0)
    print({k: v for k, v in row.items() if k != "pressure_profile_bara"})
for expected in baselines["pipeline_refinement"]:
    row = pipeline_case(jneqsim, expected["diameter_m"], expected["increments"])
    assert abs(row["pressure_drop_bar"] - expected["pressure_drop_bar"]) < 1e-6
    assert abs(row["outlet_temperature_C"] - expected["outlet_temperature_C"]) < 1e-6
    assert np.allclose(row["pressure_profile_bara"], expected["pressure_profile_bara"], atol=1e-6, rtol=0)
    print({"increments": row["increments"], "pressure_drop_bar": row["pressure_drop_bar"]})


{'diameter_m': 0.15, 'increments': 20, 'outlet_pressure_bara': 59.02951269414007, 'pressure_drop_bar': 0.9704873058599333, 'outlet_temperature_C': 30.0}
{'diameter_m': 0.2, 'increments': 20, 'outlet_pressure_bara': 59.772269786680546, 'pressure_drop_bar': 0.22773021331945387, 'outlet_temperature_C': 30.0}
{'diameter_m': 0.25, 'increments': 20, 'outlet_pressure_bara': 59.924942470216195, 'pressure_drop_bar': 0.07505752978380542, 'outlet_temperature_C': 30.0}


{'diameter_m': 0.3, 'increments': 20, 'outlet_pressure_bara': 59.969477125582145, 'pressure_drop_bar': 0.03052287441785495, 'outlet_temperature_C': 30.0}
{'increments': 10, 'pressure_drop_bar': 0.22770584136463867}
{'increments': 20, 'pressure_drop_bar': 0.22773021331945387}
{'increments': 40, 'pressure_drop_bar': 0.22774240310170768}


In [4]:
from verify_examples import COMPOSITION
for reference in baselines["hydrate_screening"]:
    composition = dict(COMPOSITION)
    composition["water"] = 0.01
    fluid = make_fluid(jneqsim, reference["pressure_bara"], 283.15, composition=composition)
    fluid.setHydrateCheck(True)
    jneqsim.thermodynamicoperations.ThermodynamicOperations(fluid).hydrateFormationTemperature()
    calculated = float(fluid.getTemperature("C"))
    assert abs(calculated - reference["hydrate_temperature_C"]) < 1e-6
    print({"pressure_bara": reference["pressure_bara"], "hydrate_temperature_C": calculated})


{'pressure_bara': 40.0, 'hydrate_temperature_C': 15.318198606719534}
{'pressure_bara': 60.0, 'hydrate_temperature_C': 18.28031025364527}
{'pressure_bara': 80.0, 'hydrate_temperature_C': 20.14687411329487}
{'pressure_bara': 100.0, 'hydrate_temperature_C': 21.43412501125664}


In [5]:
generate_chapter("ch11")
print("Chapter illustrations regenerated from maintained source and verified results.")

Chapter illustrations regenerated from maintained source and verified results.


![Hydrate boundary](../figures/hydrate_boundary.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.

![Pipeline profiles](../figures/pipeline_profiles.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.

![Pipeline workspace](../figures/pipeline_workspace.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.